Install Dependencies

In [127]:
!pip install -r requirements.txt



Prepare Environment and Parameters

In [128]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(override=True)

# Access variables
doc_intelligence_endpoint = os.getenv("DOC_INTELLIGENCE_ENDPOINT")
doc_intelligence_key = os.getenv("DOC_INTELLIGENCE_KEY")

aoai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
aoai_key = os.getenv("AZURE_OPENAI_KEY")
aoai_version = os.getenv("AZURE_OPENAI_VERSION")
aoai_embeddings_model = os.getenv("AZUE_OPENAI_EMBEDDINGS_MODEL")

ai_search_endpoint = os.getenv("AZURE_AI_SEARCH_ENDPOINT")
ai_search_key = os.getenv("AZURE_AI_SEARCH_KEY")
ai_search_version = os.getenv("AZURE_SEARCH_API_VERSION")



In [113]:
# Parameters

doc_path="docs/CELEX_32013D0121_EN_TXT.pdf"
store_md_output =True
remove_header_footer=True
print_chunks=True

#AI Search Batch Indexing
ai_search_index_name = "content-multipage-index-new-1"
max_batch_size = 1000
max_payload_size = 16 * 1024 * 1024  # 16 MB
max_retries = 3


Read PDF and Convert it to Markdown Using Document Intelligence

In [114]:
import base64
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeResult
from types import SimpleNamespace
import json

#Prepare document intelligence client
document_intelligence_client  = DocumentIntelligenceClient(
    endpoint=doc_intelligence_endpoint, credential=AzureKeyCredential(doc_intelligence_key)
)

print("Reading PDF File")
with open(doc_path, "rb") as f:
    analyze_request = {
        "base64Source": base64.b64encode(f.read()).decode("utf-8")
    }
    print("Calling Document Intelligence client")
    poller = document_intelligence_client.begin_analyze_document(
        "prebuilt-layout", 
        body=analyze_request,
        output_content_format="markdown"
    )
result= poller.result()
print("Converted PDF to Markdown succesfully")


Reading PDF File
Calling Document Intelligence client
Converted PDF to Markdown succesfully


Extract title & respective paragraphs and break it down in chunks

In [115]:
chunks = []
current_heading = None
current_paragraphs = []
current_pages = set()
chunk_id = 1

if remove_header_footer:
    print("Skipping header and footer")

for paragraph in result['paragraphs']:
    role = paragraph.get("role", "")
    text = paragraph.get("content", "")
    page = paragraph.get("boundingRegions", [{}])[0].get("pageNumber", None)
    
    if remove_header_footer and (paragraph.role=="pageHeader" or paragraph.role=="pageFooter" or paragraph.role=="pageNumber"):
        pass  
    elif role in ("sectionHeading", "title"):
        if current_heading is not None:
            chunks.append({
                "chunk_id": chunk_id,
                "heading": current_heading,
                "content": ' '.join(current_paragraphs),
                "pages": sorted(current_pages)
            })
            chunk_id += 1

        current_heading = text
        current_paragraphs = []
        current_pages = set()
        if page:
            current_pages.add(page)
    else:
        current_paragraphs.append(text)
        if page:
            current_pages.add(page)

# Final chunk
if current_heading is not None:
    chunks.append({
        "chunk_id": chunk_id,
        "heading": current_heading,
        "content": ' '.join(current_paragraphs),
        "pages": sorted(current_pages)
    })

print(f"The data has been divided into {len(chunks)} chunks")

if print_chunks:
    for i in chunks:
        print(i)

Skipping header and footer
The data has been divided into 59 chunks
{'chunk_id': 1, 'heading': 'COMMISSION DECISION of 7 March 2013', 'content': "on the safety requirements to be met by European standards for certain seats for children pursuant to Directive 2001/95/EC of the European Parliament and of the Council on general product safety (Text with EEA relevance) (2013/121/EU) THE EUROPEAN COMMISSION, Having regard to the Treaty on the Functioning of the European Union, Having regard to Directive 2001/95/EC of the European Parliament and of the Council of 3 December 2001 on general product safety (1), and in particular Article 4(1)(a) thereof, Whereas: (1) Products which conform to national standards that transpose European standards drawn up under Directive 2001/95/EC and referenced in the Official Journal of the European Union benefit from a presumption of safety. (2) European standards are to be drawn up on the basis of requirements intended to ensure that products which conform to

Preparing Embeddings for the content using Azure OPEN AI Embeddings Model

In [116]:
import openai

aoai_client = openai.AzureOpenAI(
    azure_endpoint=aoai_endpoint,
    api_key=aoai_key,
    api_version=aoai_version
)

docs = []
for chunk in chunks:
    docs.append({
        "@search.action": "upload",
        "id": str(chunk['chunk_id']),
        "page_numbers": ", ".join(str(p) for p in chunk['pages']),
        "chunk": chunk['content'],
        "section_heading": chunk['heading'],
        "chunkVector": get_embedding(aoai_client, aoai_embeddings_model, "text" + chunk['content'])
    })
                         

Create Azure AI Search Index

In [117]:
import json
import requests 

headers = {'Content-Type': 'application/json','api-key': ai_search_key}
params = {'api-version': ai_search_version }

index_payload_configuration = {
    "name": ai_search_index_name,
    "vectorSearch": {
        "algorithms": [ 
             {
                 "name": "my-hnsw-config-1",
                 "kind": "hnsw",
                 "hnswParameters": {
                     "m": 4,
                     "efConstruction": 400,
                     "efSearch": 500,
                     "metric": "cosine"
                 }
             }
        ],
        "vectorizers": [
            {
                "name": "openai",
                "kind": "azureOpenAI",
                "azureOpenAIParameters":
                {
                    "resourceUri" : aoai_endpoint,
                    "apiKey" : aoai_key,
                    "deploymentId" : aoai_embeddings_model,
                    "modelName" : aoai_embeddings_model
                }
            }
        ],
        "profiles": [  # profiles is the diferent kind of combinations of algos and vectorizers
            {
             "name": "my-vector-profile-1",
             "algorithm": "my-hnsw-config-1",
             "vectorizer":"openai"
            }
        ]
    },
    "semantic": {
        "configurations": [
            {
                "name": "my-semantic-config",
                "prioritizedFields": {
                    "titleField": {
                        "fieldName": "section_heading"
                    },
                    "prioritizedContentFields": [
                        {
                            "fieldName": "chunk"
                        }
                    ],
                    "prioritizedKeywordsFields": []
                }
            }
        ]
    },
    "fields": [
        {"name": "id", "type": "Edm.String", "key": "true", "filterable": "true" },
        {"name": "title","type": "Edm.String","searchable": "true","retrievable": "true"},
        {"name": "page_numbers","type": "Edm.String","searchable": "true","retrievable": "true"},
        {"name": "chunk","type": "Edm.String","searchable": "true","retrievable": "true"},
        {"name": "name", "type": "Edm.String", "searchable": "true", "retrievable": "true", "sortable": "false", "filterable": "false", "facetable": "false"},
        {"name": "location", "type": "Edm.String", "searchable": "false", "retrievable": "true", "sortable": "false", "filterable": "false", "facetable": "false"},
        {"name": "section_heading","type": "Edm.String","searchable": "false","retrievable": "true"},
        {
            "name": "chunkVector",
            "type": "Collection(Edm.Single)",
            "dimensions": 3072,
            "vectorSearchProfile": "my-vector-profile-1", 
            "searchable": "true",
            "retrievable": "true",
            "filterable": "false",
            "sortable": "false",
            "facetable": "false"
        }
        
    ],
}

r = requests.put(ai_search_endpoint+ "/indexes/" + ai_search_index_name,
                 data=json.dumps(index_payload_configuration), headers=headers, params=params)

if r.ok:
    print("Succesfully created AI Search Index with provided configuration")


Succesfully created AI Search Index with provided configuration


Upload Documents to AI Search

In [118]:
upload_documents_to_azure_search(docs, ai_search_index_name, ai_search_endpoint, headers, params)

Batch of 59 documents uploaded.
Upload complete: 59 documents.


In [119]:
QUESTION = "summarize potential safety hazards for children chairs"
SELECT_COLUMNS='title,chunk,page_numbers'

In [120]:
from azure.search.documents import SearchClient

credential = AzureKeyCredential(ai_search_key)
search_client_wizard = SearchClient(
    endpoint=ai_search_endpoint, index_name=ai_search_index_name, credential=credential
)

# Runs a semantic query (runs a BM25-ranked query and promotes the most relevant matches to the top)
results_wizard =  search_client_wizard.search(query_type='semantic', semantic_configuration_name='my-semantic-config',
    search_text=QUESTION, 
    select=SELECT_COLUMNS,
    top=5)

for result in results_wizard:
    print(result["@search.reranker_score"])
    print(result["title"])
    print(result["chunk"])
    print(result["page_numbers"])


3.0378856658935547
None
Entrapment hazards from gaps and openings Chair-mounted seats must be designed and manufactured to prevent the entrapment of any part of a child's body. Hazards associated with adjusting the height of the chair-mounted seat Chair-mounted seats in which the height of the sitting area can be adjusted must have locking mechanism(s) to secure the chair-mounted seat in its position of normal use. The unintentional release of locking mechanism(s) must be prevented. Hazards associated with moving parts Once the chair-mounted seat is set up for normal use, there must not be any accessible compression or shear points as a result of moving the chair-mounted seat or any part of it, the child's shifting its body weight while in the chair-mounted seat, or the application of an external force (either by another child or, unintentionally, by the carer, or by a powered mechanism). Chair-mounted seats designed to fold must have a folding mechanism that a child cannot operate and

UTILS

In [121]:
def get_embedding(client, model, input_chunk):  
    embeddings = client.embeddings.create(  
        model=model,  
        input=input_chunk  
    )  
    return embeddings.data[0].embedding  

In [122]:
import json
import requests
import time

def upload_documents_to_azure_search(docs, index_name, azure_search_endpoint, headers, params):
    def batch_generator(items):
        batch = []
        current_size = 0

        for item in items:
            item_str = json.dumps(item)
            item_bytes = len(item_str.encode('utf-8'))

            if len(batch) >= max_batch_size or current_size + item_bytes > max_payload_size:
                yield batch
                batch = []
                current_size = 0

            batch.append(item)
            current_size += item_bytes

        if batch:
            yield batch

    def send_batch(batch, attempt=1):
        payload = {"value": batch}
        response = requests.post(
            f"{azure_search_endpoint}/indexes/{index_name}/docs/index",
            headers=headers,
            params=params,
            data=json.dumps(payload)
        )

        if response.status_code == 200:
            print(f"Batch of {len(batch)} documents uploaded.")
        else:
            print(f"Batch failed (attempt {attempt}): {response.status_code}")
            print(response.text)

            if attempt < max_retries:
                time.sleep(2 ** attempt)  # Exponential backoff
                send_batch(batch, attempt + 1)
            else:
                print("Max retries reached. Skipping batch.")

    total_uploaded = 0
    for batch in batch_generator(docs):
        send_batch(batch)
        total_uploaded += len(batch)

    print(f"Upload complete: {total_uploaded} documents.")

